In [ ]:
# ========== PART 1: Full Project Analysis ==========

import os
import yaml
import pandas as pd

# === CONFIG ===
PROJECTS_DIR = r"C:\Users\Admin\OneDrive\Education\Master of Info - Thesis\Mobile App Data\Config Files"
OUTPUT_DIR = r"C:\GitHub\Android-Mobile-Apps"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV_1 = os.path.join(OUTPUT_DIR, "Project_List.csv")

# === CLASSIFICATION KEYWORDS ===
TEST_TYPES = {
    'firebase_Full': ['gcloud firebase test android run'],
    'firebase_Compact': ['Firebase-Test-Lab-Action'],
    'appcenter_test': ['appcenter test run', 'microsoft/appcenter-test-cli-action'],
    'browserstack_test': ['browserstack', 'browserstack/github-actions'],
    'GitHub_emulator_full': ['android-emulator-runner'],
    'GitHub_emulator_compact': ['malinskiy/action-android/emulator-run-cmd'],
    'GitHub_emulator_manual': ['create avd'],
    'GitHub_GMD': ['cleanManagedDevices'],
    'Unit_Test': ['gradlew test', './gradlew test', 'testDebugUnitTest', 'testReleaseUnitTest',
                  'test', 'run unit tests', 'run: test', 'npm test', 'yarn test'],
    'Other': ['instrumentation', 'instrument']
}

# === DETECTION LOGIC ===
def detect_testing_types(yaml_text):
    uncommented_text = '\n'.join(
        line for line in yaml_text.splitlines()
        if not line.strip().startswith('#')
    ).lower()
    found = set()
    for label, keywords in TEST_TYPES.items():
        for kw in keywords:
            if kw.lower() in uncommented_text:
                found.add(label)
    return found

def detect_ci_platform(file_path, yaml_text):
    text = yaml_text.lower()
    if ".github" in file_path.lower() or "github" in text:
        return "GitHub Actions"
    elif ".travis" in file_path.lower() or "travis" in text:
        return "Travis CI"
    elif "circleci" in file_path.lower() or "circleci" in text:
        return "CircleCI"
    elif "bitrise" in file_path.lower() or "bitrise" in text:
        return "Bitrise"
    else:
        return "Unknown"

# === PARSE YAML FILES ===
project_results = []
for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    raw = f.read().replace('\t', ' ')
                    detected = detect_testing_types(raw)
                    platform = detect_ci_platform(file_path, raw)
                    filename = os.path.basename(file_path)
                    parts = filename.split(".")
                    project_name = (parts[1] if len(parts) > 2 else parts[0]).lower()

                    project_results.append({
                        'project': project_name,
                        'test_types': ', '.join(sorted(detected)) if detected else 'none',
                        'ci_platform': platform
                    })
            except Exception:
                project_results.append({
                    'project': project_name,
                    'test_types': 'none',
                    'ci_platform': 'Error'
                })

# === BUILD FINAL DATAFRAME ===
df = pd.DataFrame(project_results)

df['Unit Test'] = df['test_types'].apply(
    lambda x: any(item.strip().startswith('Unit_Test') for item in x.split(','))
)

df['Instrumentation Testing'] = df['test_types'].apply(
    lambda x: x.strip().lower() != 'none' and not (
        len([t for t in x.split(',') if t.strip()]) == 1 and x.strip() == 'Unit_Test'
    )
)

df['GitHub Action'] = df['test_types'].apply(
    lambda x: any(item.strip().startswith('GitHub') for item in x.split(','))
)

df['GitHub Action Type'] = df['test_types'].apply(
    lambda x: ', '.join([item.strip() for item in x.split(',') if item.strip().startswith('GitHub')])
)

df['Third_Party'] = df['test_types'].apply(
    lambda x: any(
        not item.strip().startswith('GitHub') and item.strip().lower() not in ['none', 'other', 'unit_test']
        for item in x.split(',')
    )
)

df['Third_Party_Name'] = df['test_types'].apply(
    lambda x: ', '.join([
        item.strip() for item in x.split(',')
        if not item.strip().startswith('GitHub') and item.strip().lower() not in ['none', 'other', 'unit_test']
    ])
)

# === UPDATE LABELS BASED ON CI PLATFORM ===
def replace_manual_label(row):
    test_list = [item.strip() for item in row['test_types'].split(',')]
    if 'GitHub_emulator_manual' in test_list:
        test_list.remove('GitHub_emulator_manual')
        platform_prefix = row['ci_platform'].split()[0]  # GitHub, Travis, etc.
        test_list.append(f"{platform_prefix}_emulator_manual")
    return ', '.join(sorted(test_list))

df['test_types'] = df.apply(replace_manual_label, axis=1)

# === EXPORT TO CSV ===
df.to_csv(OUTPUT_CSV_1, index=False)
print(f"\n✅ Project List-Part 1 written to: {OUTPUT_CSV_1}")


# ========== PART 2: CI Platform Only ==========

OUTPUT_CSV_2 = os.path.join(OUTPUT_DIR, "YML_List.csv")
project_results_ci_only = []

for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    raw = f.read().replace('\t', ' ')
                    platform = detect_ci_platform(file_path, raw)
                    filename = os.path.basename(file_path)
                    parts = filename.split(".")
                    project_name = (parts[1] if len(parts) > 2 else parts[0]).lower()

                    project_results_ci_only.append({
                        'project': project_name,
                        'ci_platform': platform
                    })
            except Exception:
                project_results_ci_only.append({
                    'project': project_name,
                    'ci_platform': 'Error'
                })

df_ci_only = pd.DataFrame(project_results_ci_only)
df_ci_only.to_csv(OUTPUT_CSV_2, index=False)
print(f"\n✅ Project List-CI Platform Only written to: {OUTPUT_CSV_2}")
